# Uncertainty Analysis: Roll Angle
This notebook analyzes the impact of camera roll angle variations on slope estimation accuracy.
It includes:
1.  **Image Transformation**: Generating perspective views with varying Roll angles (simulated via Phi parameter).
2.  **Result Analysis**: Comparing estimated slopes against ground truth.
3.  **Visualization**: Plotting uncertainty and data availability.

In [1]:
import os
import cv2
import numpy as np
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
import seaborn as sns
from zensvi.transform import ImageTransformer

# Plotting settings
plt.rcParams.update({'font.size': 14})
plt.rcParams['font.family'] = 'Arial'

In [2]:
# Configuration & Paths
BASE_DIR = '.'

# Input Data
VALIDATION_SAMPLES_PATH = os.path.join(BASE_DIR, 'subsamples_result.gpkg')
SOURCE_PANOS_DIR = os.path.join(BASE_DIR, 'pano')

# Working Directories
DEGRADATION_DIR = os.path.join(BASE_DIR, 'roll')
TRANSFORMED_DIR = os.path.join(DEGRADATION_DIR, 'roll-panos')
RESULTS_DIR = os.path.join(DEGRADATION_DIR, 'roll-test')

# Output
FIGURE_SAVE_PATH = os.path.join(BASE_DIR, 'rolled_uncertainty.png')

# Parameters
ROLL_ANGLES = list(range(-30, 31, 5))

In [ ]:
# Load validation samples
subsamples = gpd.read_file(VALIDATION_SAMPLES_PATH)
display(subsamples.head())

In [ ]:
def run_roll_transformation(input_dir, base_output_dir, angles):
    """
    Transform panoramas with varying Roll angles.
    Note: This uses the 'phi' parameter to simulate the angle change as per the original analysis.
    """
    for angle in angles:
        output_dir = os.path.join(base_output_dir, f'chunk_{angle}_rolled')
        print(f"Processing Roll {angle}° -> {output_dir}")
        
        image_transformer = ImageTransformer(dir_input=input_dir, dir_output=output_dir)
        image_transformer.transform_images(
            style_list="perspective",
            FOV=90,
            theta=90,
            phi=angle,  
            aspects=(10, 10),
            show_size=100
        )
        
        # Cleanup
        os.system(f"find {output_dir} -name '*Direction_0*' -delete")
        os.system(f"find {output_dir} -name '*Direction_180*' -delete")

run_roll_transformation(SOURCE_PANOS_DIR, TRANSFORMED_DIR, ROLL_ANGLES)

In [3]:
## run vision2slope model on transformed images
## Import Vision2Slope modules
from vision2slope import (
    PipelineConfig,
    Vision2SlopePipeline
)

config_basic = PipelineConfig(
    input_dir="/Users/cubics/Vision2Slope/uncertainty_analysis/roll/roll-panos/chunk_-5_rolled/perspective",           # Input image directory
    output_dir=RESULTS_DIR    # Output results directory
)
pipeline_basic = Vision2SlopePipeline(config_basic)
results_basic = pipeline_basic.process_batch()

/Users/cubics/miniconda3/envs/zensvi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


INFO - Initializing pipeline components...
INFO - Loading segmentation model: facebook/mask2former-swin-large-mapillary-vistas-semantic


Using a slow image processor as `use_fast` is unset and a slow processor was saved with this model. `use_fast=True` will be the default behavior in v4.52, even if the model was saved with a slow processor. This will result in minor differences in outputs. You'll still be able to use a slow processor with `use_fast=False`.
/Users/cubics/miniconda3/envs/zensvi/lib/python3.10/site-packages/transformers/utils/deprecation.py:172: UserWarning: The following named arguments are not valid for `Mask2FormerImageProcessor.__init__` and were ignored: '_max_size'
  return func(*args, **kwargs)


INFO - Model loaded successfully on device: cpu
INFO - Pipeline components initialized successfully
INFO - Found 2 images to process


Processing images:   0%|          | 0/2 [00:00<?, ?it/s]

...

INFO - Successfully processed _3mZg7nzlR6r1SKHOwQI-A_Direction_90_FOV_90_aspect_10--10_raw.png


Processing images: 100%|██████████| 2/2 [00:06<00:00,  3.12s/it]

                                            filename                 pano_id  \
0  _3mZg7nzlR6r1SKHOwQI-A_Direction_270_FOV_90_as...  _3mZg7nzlR6r1SKHOwQI-A   
1  _3mZg7nzlR6r1SKHOwQI-A_Direction_90_FOV_90_asp...  _3mZg7nzlR6r1SKHOwQI-A   

   skew_angle  skew_confidence  num_lines_detected  correction_applied  \
0         0.0              158                 242               False   
1         0.0               33                  55               False   

  corrected_filename  road_edge_line_slope  road_edge_line_intercept  \
0               None              0.018952                972.151768   
1               None              0.001156                769.673810   

   road_edge_line_angle  road_area stage_completed   status error_message  
0              1.085715      17290        complete  success                
1              0.066244     209792        complete  success                
INFO - Results saved to: /Users/cubics/Vision2Slope/uncertainty_analysis/roll/roll-test/vis


Computing road slopes: 100%|██████████| 1/1 [00:00<00:00, 3584.88it/s]

INFO - Estimated results saved to: /Users/cubics/Vision2Slope/uncertainty_analysis/roll/roll-test/vision2slope_results_20251213_163301_estimate.csv
INFO - Intermediate results saved to: /Users/cubics/Vision2Slope/uncertainty_analysis/roll/roll-test/intermediate_results
INFO - ============================================================
INFO - VISION2SLOPE PROCESSING SUMMARY
INFO - ============================================================
INFO - Total images: 2
INFO -   success: 2 (100.0%)
INFO - 
Successful Results Statistics:
INFO -   Avg skew angle: 0.00°
INFO -   Avg road edge line slope: 0.0101
INFO -   Avg road edge line angle: 0.58°
INFO -   Avg road area: 113541 pixels
INFO -   Avg road estimated slope: 0.14°
INFO - ============================================================


In [ ]:
def search_csv_files(directory, keyword=None):
    csv_files = []
    for root, dirs, files in os.walk(directory):
        for file in files:
            if file.endswith('.csv'):
                if keyword is None or keyword in file:
                    csv_files.append(os.path.join(root, file))
    return csv_files

def load_roll_results(base_path, angles, sample_df):
    """Load and merge results from different Roll runs."""
    merged_df = sample_df[['pano_id', 'slope_1m', 'slope_10m', 'slope_30m']].copy()
    merged_df = merged_df.set_index('pano_id')
    
    available_ratios = []
    
    for angle in angles:
        # Note: Folder naming convention from original notebook: chunk_{angle}Rolled
        search_path = os.path.join(base_path, f'chunk_{angle}Rolled')
        csv_files = search_csv_files(search_path, keyword='adjusted')
        
        if csv_files:
            # Use the first found CSV
            res_df = pd.read_csv(csv_files[0])
            
            # Calculate availability
            n = len(res_df)
            available_ratios.append(n / 2000 * 100) # Assuming 2000 total samples
            
            res_df.set_index('pano_id', inplace=True)
            res_df = res_df[['adjusted_angle']]
            res_df = res_df.rename(columns={'adjusted_angle': f'rolled_{angle}_adjusted_angle'})
            
            # Drop duplicates
            res_df = res_df[~res_df.index.duplicated(keep='first')]
            
            merged_df = merged_df.join(res_df, how='left')
            
            # Calculate difference
            merged_df[f'rolled_{angle}_adjusted_angle_diff'] = merged_df[f'rolled_{angle}_adjusted_angle'] - merged_df['slope_1m']
        else:
            print(f"No results found for Roll {angle}°")
            available_ratios.append(0)
            
    return merged_df, available_ratios

# Run analysis
analyzed_df, availability = load_roll_results(RESULTS_DIR, ROLL_ANGLES, subsamples)
display(analyzed_df.head())

In [ ]:
def plot_roll_summary(sample_df, availability_pct, angles, save_path=None):
    import matplotlib.ticker as mticker
    
    angles = list(angles)
    plot_angles = []
    rename_map = {}
    for angle in angles:
        col = f'rolled_{angle}_adjusted_angle_diff'
        if col in sample_df.columns:
            plot_angles.append(angle)
            rename_map[col] = f"{angle}°"

    if not plot_angles:
        print("No data to plot.")
        return

    order_labels = [f"{angle}°" for angle in plot_angles]
    plot_df = (
        sample_df[[f'rolled_{angle}_adjusted_angle_diff' for angle in plot_angles]]
        .rename(columns=rename_map)
        .melt(var_name='Roll angle (°)', value_name='Slope difference (°)')
    )
    plot_df['Roll angle (°)'] = pd.Categorical(plot_df['Roll angle (°)'], categories=order_labels, ordered=True)

    # Create figure
    fig = plt.figure(constrained_layout=False, figsize=(12, 8))
    gs = fig.add_gridspec(2, 1, height_ratios=[3, 1], hspace=0.15)
    
    # Boxplot
    highlight_color = sns.color_palette('crest', n_colors=7)[-2]
    ax_box = fig.add_subplot(gs[0])
    sns.boxplot(
        data=plot_df,
        x='Roll angle (°)',
        y='Slope difference (°)',
        ax=ax_box,
        width=0.3,
        fliersize=2.2,
        linewidth=1.2,
        medianprops={'color': '#0F3460', 'linewidth': 2},
        whiskerprops={'color': '#0F3460', 'linewidth': 1.2},
        capprops={'color': '#0F3460', 'linewidth': 1.2},
        showfliers=False,
        showcaps=False
    )
    
    for patch in ax_box.artists:
        patch.set_alpha(0.38)
        patch.set_edgecolor('#0F3460')
        patch.set_facecolor('none')

    ax_box.axhline(0, color='#5B6270', linestyle='--', linewidth=1, alpha=0.8, zorder=0)
    ax_box.set_xlabel('')
    ax_box.set_ylabel('Slope difference (°)', labelpad=12)
    ax_box.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.4)
    ax_box.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f'{x:.0f}'))
    
    # Remove x-axis labels for boxplot
    ax_box.set_xticklabels([])
    ax_box.tick_params(axis='x', which='both', length=0, labelbottom=False)

    # Availability Line Plot
    positions = np.arange(len(order_labels))
    
    ax_line = fig.add_subplot(gs[1], sharex=ax_box)
    ax_line.plot(positions, availability_pct, color=highlight_color, marker='o', linewidth=2.4, label='Availability')
    ax_line.fill_between(positions, availability_pct, color=highlight_color, alpha=0.15)
    
    ax_line.set_ylabel('Available imagery (%)', labelpad=12)
    ax_line.set_xlabel('Roll angle (°)', labelpad=12)
    ax_line.set_xticks(positions)
    ax_line.set_xticklabels(order_labels)
    ax_line.set_ylim(min(min(availability_pct), 60) * 0.7, max(105, max(availability_pct) * 1.05))
    ax_line.grid(axis='y', linestyle='--', linewidth=0.6, alpha=0.35)

    if save_path:
        fig.savefig(save_path, dpi=300, bbox_inches='tight')
    plt.show()

# Plot
plot_roll_summary(
    analyzed_df,
    availability,
    ROLL_ANGLES,
    save_path=FIGURE_SAVE_PATH
)